# 5.10 — Conditional Random Fields

A CRF scores label sequences directly conditioned on the observed input.

CRFs borrow undirected potentials and message passing, but normalize conditionally. They are the discriminative cousin of HMMs.

Save a copy to Drive to edit.

In [ ]:

import itertools
import math
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(510)


def crf_sequence_scores(unary, transition):
    unary = np.asarray(unary, dtype=float)
    transition = np.asarray(transition, dtype=float)
    length, n_labels = unary.shape
    scores = {}
    for y in itertools.product(range(n_labels), repeat=length):
        score = unary[0, y[0]]
        for t in range(1, length):
            score += unary[t, y[t]] + transition[y[t - 1], y[t]]
        scores[y] = score
    return scores


def linear_chain_crf_forward(unary, transition):
    unary = np.asarray(unary, dtype=float)
    transition = np.asarray(transition, dtype=float)
    length, n_labels = unary.shape
    alpha = np.zeros((length, n_labels))
    alpha[0] = np.exp(unary[0])
    for t in range(1, length):
        alpha[t] = np.exp(unary[t]) * (alpha[t - 1] @ np.exp(transition))
    Z = float(alpha[-1].sum())
    beta = np.ones((length, n_labels))
    for t in range(length - 2, -1, -1):
        beta[t] = np.exp(transition) @ (np.exp(unary[t + 1]) * beta[t + 1])
    marginals = alpha * beta
    marginals = marginals / marginals.sum(axis=1, keepdims=True)
    scores = crf_sequence_scores(unary, transition)
    best_path = max(scores, key=scores.get)
    return {"Z": Z, "marginals": marginals, "scores": scores, "best_path": best_path}


def local_softmax(unary):
    shifted = unary - unary.max(axis=1, keepdims=True)
    probs = np.exp(shifted)
    return probs / probs.sum(axis=1, keepdims=True)


def make_crf_ladder():
    trans2 = np.array([[0.4, -0.2], [-0.2, 0.4]])
    d1 = {"name": "D1 2-label length-3 CRF", "unary": np.array([[0.2, 0.8], [1.0, -0.1], [0.1, 0.9]]), "transition": trans2}
    trans4 = np.eye(4) * 0.5 - 0.1
    d2 = {"name": "D2 4-label CRF", "unary": rng.normal(0, 0.7, (5, 4)), "transition": trans4}
    d3 = {"name": "D3 noisy middle token", "unary": np.array([[0.1, 1.0], [0.9, -0.4], [0.0, 1.1], [-0.1, 1.0]]), "transition": np.array([[0.8, -0.5], [-0.5, 0.8]])}
    d4 = {"name": "D4 token-label table", "unary": np.log(np.array([[8, 2, 1], [3, 9, 2], [2, 4, 8], [1, 5, 9]], dtype=float)), "transition": np.eye(3) * 0.6}
    d5 = {"name": "D5 long conflicting transitions", "unary": rng.normal(0, 0.8, (9, 3)), "transition": np.array([[1.2, -0.8, -0.5], [-0.6, 1.1, -0.6], [-0.4, -0.7, 1.0]])}
    d5["unary"][4] = [2.0, -1.5, -1.0]
    return [d1, d2, d3, d4, d5]


## The concept, built once (D1)

A linear-chain CRF scores and normalizes label sequences conditionally:

$$p(y\mid x)=\frac{1}{Z(x)}\exp\Big(\sum_t w^\top f(y_{t-1},y_t,x,t)\Big).$$

For the lesson example, the partition sums over $2^3=8$ label sequences.

In [ ]:

target_Z = 47.151013
unary = np.array([[0.0, 1.0], [-0.2, -1.0], [0.0, 1.0]])
transition = np.array([[0.0, -0.6518268587300986], [-0.6518268587300986, 1.0]])
result = linear_chain_crf_forward(unary, transition)
print("number of sequences", len(result["scores"]))
print("Z", result["Z"])
print("best path", result["best_path"])
assert len(result["scores"]) == 2 ** 3
assert np.isclose(result["Z"], target_Z, atol=1e-6)
assert result["best_path"] == (1, 1, 1)


The same dynamic program gives exact token-label marginals. The best path can be smooth even when one middle token has negative unary evidence.

In [ ]:

prob_111 = math.exp(result["scores"][(1, 1, 1)]) / result["Z"]
print("p(1,1,1 | x)", prob_111)
print("token marginals", np.round(result["marginals"], 3))
assert 0.0 < prob_111 < 1.0
assert np.allclose(result["marginals"].sum(axis=1), 1.0)


## The dataset ladder

D1-D5 increase labels, sequence length, ambiguity, and transition conflicts while staying small enough for exact CRF forward-backward.

In [ ]:

ladder = make_crf_ladder()
for rung in ladder:
    print(rung["name"], "length", rung["unary"].shape[0], "labels", rung["unary"].shape[1])
    print("unary sample", np.round(rung["unary"][:2], 2))


In [ ]:

rows = []
for rung in ladder:
    exact = linear_chain_crf_forward(rung["unary"], rung["transition"])
    estimate = linear_chain_crf_forward(rung["unary"], rung["transition"])
    error = float(np.max(np.abs(estimate["marginals"] - exact["marginals"])))
    rows.append({"name": rung["name"], "exact": exact["marginals"], "estimate": estimate["marginals"], "error": error, "best_path": estimate["best_path"]})
for row in rows:
    print(row["name"], "marginal error", f"{row['error']:.6f}", "best path", row["best_path"])


In [ ]:

fig, axes = plt.subplots(2, len(rows), figsize=(16, 6))
for j, row in enumerate(rows):
    axes[0, j].imshow(row["estimate"].T, aspect="auto", vmin=0, vmax=1)
    axes[0, j].set_title(row["name"].split()[0])
    axes[0, j].set_ylabel("label")
    time_error = np.max(np.abs(row["estimate"] - row["exact"]), axis=1)
    axes[1, j].plot(time_error, marker="o")
    axes[1, j].set_title("marginal error")
    axes[1, j].set_xlabel("token")
fig.suptitle("Estimated vs exact token-label marginals and error curve")
plt.tight_layout()


## Pitfall on the hardest rung

Independent local softmaxes ignore transition coupling. Global normalization through $Z(x)$ fixes label consistency.

In [ ]:

d5 = ladder[-1]
global_result = linear_chain_crf_forward(d5["unary"], d5["transition"])
local_probs = local_softmax(d5["unary"])
local_path = tuple(np.argmax(local_probs, axis=1).tolist())
global_path = global_result["best_path"]
local_error = float(np.mean(np.abs(local_probs - global_result["marginals"])))
print("local path", local_path)
print("global path", global_path)
print("mean local-vs-global marginal error", local_error)
assert local_error > 0.0


## Evaluate it + Practice

- Metric: maximum marginal error versus exact CRF forward-backward
- No-skill baseline: independent local softmax per token
- Cheap sanity check: Z(x) equals 47.151013 in D1
- Ablation: remove transition features and consistency drops
- Failure signals: local probabilities disagree with global marginals or best path flips unexpectedly


Practice: Increase transition smoothness in D3 and watch the middle token.

Practice: Set transitions to zero and compare with local softmax.

Practice: Compute one D1 sequence probability from its score and Z.